# Positional Embeddings in Language Models

This notebook is an introduction to positional information in Transformers.

We will cover four ideas:
1. Absolute positional encoding (sinusoidal)
2. Relative positional encoding (distance-based bias)
3. Rotary positional embedding (RoPE)
4. No positional encoding (NoPE)

In [14]:
from IPython.display import HTML, display
colab_button = HTML(
    '<a target="_blank" href="https://colab.research.google.com/github/surrey-nlp/NLP-2026/blob/main/lab06/lab06-Positional-Encoding-in-LM.ipynb">'
    '<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>'
)
display(colab_button)

## Installation

In [ ]:
%pip install numpy matplotlib torch

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

np.set_printoptions(precision=4, suppress=True)
torch.set_printoptions(precision=4, sci_mode=False)

## Why Positional Information?

Self-attention looks at tokens in parallel. Without extra information, the model sees a sentence like a bag of words.

Example:
- "dog bites man"
- "man bites dog"

Same words, different meaning. Positional methods tell the model about order.

## 1) Absolute Positional Encoding (Sinusoidal)

Simple idea: each position gets its own vector.
Then we add this vector to the token embedding at that position.

Classic formula (Vaswani et al., 2017):
- PE(pos, 2i)   = sin(pos / 10000^(2i / d_model))
- PE(pos, 2i+1) = cos(pos / 10000^(2i / d_model))

In [ ]:
def sinusoidal_pe(seq_len, d_model, base=10000):
    """Create sinusoidal positional encodings with shape [seq_len, d_model]."""
    if d_model % 2 != 0:
        raise ValueError('d_model must be even for sin/cos pairs.')

    # Each row is a position index: 0, 1, 2, ...
    positions = np.arange(seq_len)[:, None]  # [seq_len, 1]
    # i picks sin/cos frequency bands (one frequency per pair of dimensions).
    i = np.arange(d_model // 2)[None, :]     # [1, d_model/2]
    angle_rates = 1 / np.power(base, (2 * i) / d_model)
    angles = positions * angle_rates

    # Fill even dimensions with sin, odd dimensions with cos.
    pe = np.zeros((seq_len, d_model))
    pe[:, 0::2] = np.sin(angles)
    pe[:, 1::2] = np.cos(angles)
    return pe

# Quick sanity check: small matrix with first few rows/columns printed.
pe = sinusoidal_pe(seq_len=8, d_model=16)
print('Shape:', pe.shape)
print(pe[:3, :8])

In [ ]:
plt.figure(figsize=(8, 4))
plt.imshow(sinusoidal_pe(seq_len=64, d_model=64), aspect='auto', cmap='viridis')
plt.colorbar()
plt.title('Absolute Sinusoidal Positional Encoding Matrix')
plt.xlabel('Embedding dimension')
plt.ylabel('Position index')
plt.show()

### Absolute encoding heatmap

- Each row is one token position; each column is one embedding dimension.
- Bright/dark stripes show sinusoidal patterns across positions.
- Left-side dimensions change faster (higher frequency), while right-side dimensions vary more slowly (lower frequency).
- This mix of fast and slow patterns helps encode both nearby and long-range order.

### Quick intuition

Lower dimensions change faster and higher dimensions change slower.
So one vector carries both local and long-range position patterns.

In [ ]:
def cosine_similarity(a, b):
    # Cosine similarity tells us if two vectors point in similar directions.
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12)

pe = sinusoidal_pe(seq_len=32, d_model=32)

# Compare adjacent positions (0 and 1) in low-frequency and high-frequency subspaces.
sim_low = cosine_similarity(pe[0, :6], pe[1, :6])
sim_high = cosine_similarity(pe[0, 26:], pe[1, 26:])

print('Cosine similarity on low dims  (positions 0 vs 1):', round(sim_low, 4))
print('Cosine similarity on high dims (positions 0 vs 1):', round(sim_high, 4))

## 2) Relative Positional Encoding (Distance-Based)

Instead of storing absolute index (0, 1, 2, ...), we focus on distance between tokens (j - i).

In simple terms: the model cares about *how far apart* two tokens are, not only their absolute positions.

A common practical version is to add a distance bias to attention scores.

In [ ]:
def relative_distance_bias(seq_len, max_dist=8):
    """Create a clipped distance matrix of shape [seq_len, seq_len]."""
    i = np.arange(seq_len)[:, None]
    j = np.arange(seq_len)[None, :]
    distances = j - i
    return np.clip(distances, -max_dist, max_dist)

seq_len = 12
bias = relative_distance_bias(seq_len, max_dist=5)

plt.figure(figsize=(5, 4))
plt.imshow(bias, cmap='coolwarm')
plt.colorbar()
plt.title('Relative Distance Bias Matrix (j - i)')
plt.xlabel('Key position j')
plt.ylabel('Query position i')
plt.show()

print(bias[:5, :5])

### Relative distance bias

- Cell (i, j) stores the relative distance from query position i to key position j.
- The diagonal is zero distance (a token compared with itself).
- Values become more positive/negative as positions are farther apart.
- In practice, this kind of matrix is converted into attention bias terms so distance affects attention scores.

### Add relative bias to toy attention

Here we compare attention weights with and without a relative bias term.

In [ ]:
def softmax(x, axis=-1):
    # Stable softmax: subtract max to avoid overflow in exp().
    x = x - np.max(x, axis=axis, keepdims=True)
    ex = np.exp(x)
    return ex / np.sum(ex, axis=axis, keepdims=True)

np.random.seed(7)
L, d = 8, 16
Q = np.random.randn(L, d)
K = np.random.randn(L, d)

# Base attention scores from query-key dot products.
scores_plain = (Q @ K.T) / np.sqrt(d)
attn_plain = softmax(scores_plain, axis=-1)

# Relative-style recency bias: larger distance => larger penalty.
dist = np.abs(np.arange(L)[:, None] - np.arange(L)[None, :])
scores_rel = scores_plain - 0.35 * dist
attn_rel = softmax(scores_rel, axis=-1)

print('Attention row for query token 3 (plain):')
print(np.round(attn_plain[3], 3))
print('Attention row for query token 3 (with relative bias):')
print(np.round(attn_rel[3], 3))

## 3) Rotary Positional Embedding (RoPE)

RoPE does not add a position vector to token embeddings.
Instead, it rotates query and key vectors in 2D pairs based on position.

Key intuition: this makes attention naturally depend on relative position differences.

In [ ]:
def rope_angles(seq_len, head_dim, base=10000.0):
    if head_dim % 2 != 0:
        raise ValueError('head_dim must be even for RoPE.')
    half = head_dim // 2
    positions = torch.arange(seq_len).float().unsqueeze(1)  # [L, 1]
    freqs = torch.arange(half).float().unsqueeze(0)         # [1, half]
    # Inverse frequencies control how fast each 2D pair rotates.
    inv_freq = 1.0 / (base ** (2 * freqs / head_dim))
    return positions * inv_freq  # [L, half]

def apply_rope(x):
    """x shape: [seq_len, head_dim]"""
    seq_len, head_dim = x.shape
    angles = rope_angles(seq_len, head_dim).to(x.device)
    cos = torch.cos(angles)
    sin = torch.sin(angles)

    # Split into even/odd channels so each pair forms one 2D vector.
    x_even = x[:, 0::2]
    x_odd = x[:, 1::2]

    # Rotate each 2D pair by a position-dependent angle.
    rot_even = x_even * cos - x_odd * sin
    rot_odd = x_even * sin + x_odd * cos

    # Merge rotated pairs back to original layout.
    out = torch.zeros_like(x)
    out[:, 0::2] = rot_even
    out[:, 1::2] = rot_odd
    return out

torch.manual_seed(0)
x = torch.randn(6, 8)
x_rope = apply_rope(x)

print('Input shape:', x.shape)
print('RoPE output shape:', x_rope.shape)
print('First vector (before):', x[0])
print('First vector (after): ', x_rope[0])

In [ ]:
# Rotation should preserve the length of each 2D pair (up to tiny floating-point error).
before_pair_norm = torch.sqrt(x[:, 0::2] ** 2 + x[:, 1::2] ** 2)
after_pair_norm = torch.sqrt(x_rope[:, 0::2] ** 2 + x_rope[:, 1::2] ** 2)
max_diff = torch.max(torch.abs(before_pair_norm - after_pair_norm)).item()
print('Max pair-norm difference:', max_diff)

## 4) No Positional Encoding (NoPE)

NoPE means we do not inject explicit position information.

This can still work in some settings, but word order may be harder for the model to learn, especially for long-range structure.

In [ ]:
# Tiny toy example: compare attention with NoPE vs absolute sinusoidal PE.
np.random.seed(42)
L, d = 6, 12
token_embeddings = np.random.randn(L, d)

# NoPE path: raw token embeddings only.
scores_nope = (token_embeddings @ token_embeddings.T) / np.sqrt(d)
attn_nope = softmax(scores_nope, axis=-1)

# Absolute PE path: add positional vectors before computing attention scores.
pe_small = sinusoidal_pe(seq_len=L, d_model=d)
x_with_abs = token_embeddings + pe_small
scores_abs = (x_with_abs @ x_with_abs.T) / np.sqrt(d)
attn_abs = softmax(scores_abs, axis=-1)

print('NoPE attention row for token 2:')
print(np.round(attn_nope[2], 3))
print('Absolute-PE attention row for token 2:')
print(np.round(attn_abs[2], 3))

## Mini Demo: Toggle Positional Methods on the Same Toy Sentence

This mini experiment keeps token embeddings and projection matrices fixed, then changes only the positional method.

You can compare attention maps for:
- NoPE
- Absolute
- Relative (distance penalty)
- RoPE

Goal: see how different positional choices reshape attention, even with the same tokens.

In [ ]:
# Tiny toy LM-style demo: compare attention under NoPE, Absolute, Relative, and RoPE.
tokens = ['the', 'cat', 'sat', 'on', 'the', 'mat']
L = len(tokens)
d_model = 16

np.random.seed(123)
token_emb = np.random.randn(L, d_model)

# Shared projections so comparisons are fair across methods
Wq = np.random.randn(d_model, d_model) / np.sqrt(d_model)
Wk = np.random.randn(d_model, d_model) / np.sqrt(d_model)

def get_attention_weights(x, mode='nope', rel_strength=0.35):
    x_in = x.copy()

    if mode == 'absolute':
        x_in = x_in + sinusoidal_pe(seq_len=L, d_model=d_model)

    q = x_in @ Wq
    k = x_in @ Wk

    if mode == 'rope':
        q_t = torch.tensor(q, dtype=torch.float32)
        k_t = torch.tensor(k, dtype=torch.float32)
        q = apply_rope(q_t).numpy()
        k = apply_rope(k_t).numpy()

    scores = (q @ k.T) / np.sqrt(d_model)

    if mode == 'relative':
        dist = np.abs(np.arange(L)[:, None] - np.arange(L)[None, :])
        scores = scores - rel_strength * dist

    return softmax(scores, axis=-1)

modes = ['nope', 'absolute', 'relative', 'rope']
attn_by_mode = {m: get_attention_weights(token_emb, mode=m) for m in modes}

query_index = 2  # token 'sat'
print('Tokens:', tokens)
print("\nAttention from query token 'sat' (index 2):")
for m in modes:
    print(f"{m:>8}:", np.round(attn_by_mode[m][query_index], 3))

max_val = max(np.max(attn_by_mode[m]) for m in modes)
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), sharey=True, constrained_layout=True)
for ax, m in zip(axes, modes):
    im = ax.imshow(attn_by_mode[m], vmin=0, vmax=max_val, cmap='magma')
    ax.set_title(m.upper())
    ax.set_xticks(range(L))
    ax.set_yticks(range(L))
    ax.set_xticklabels(tokens, rotation=45, ha='right')
    ax.set_yticklabels(tokens)
fig.colorbar(im, ax=axes.ravel().tolist(), fraction=0.02)
plt.suptitle('Toy Attention Patterns by Positional Method')
plt.show()

### 4-way comparison

- All four panels use the same token embeddings and projection matrices.
- The only thing that changes is the positional method, so differences are due to position handling.
- Brighter cells mean higher attention weight from row token (query) to column token (key).
- Compare row-by-row to see how each positional method reshapes where attention is concentrated.

## Summary Table

| Method | How position is injected | Main idea |
|---|---|---|
| Absolute (sin/cos) | Add vector to embeddings | Position index has fixed code |
| Relative | Add distance-based term to attention | Distance between tokens matters |
| RoPE | Rotate Q and K vectors | Relative offsets appear in dot products |
| NoPE | No explicit signal | Model relies on content and training biases |

## Exercises

1. Change `d_model` and observe the sinusoidal heatmap.
2. Increase sequence length and inspect whether relative bias emphasizes nearby tokens more strongly.
3. In the RoPE cell, change `base` from 10000 to 1000 and compare outputs.
4. Design one sentence pair where NoPE should struggle more than PE-based methods.

## References

1. Vaswani, A. et al. (2017). *Attention Is All You Need*. https://arxiv.org/abs/1706.03762
2. Shaw, P., Uszkoreit, J., Vaswani, A. (2018). *Self-Attention with Relative Position Representations*. https://arxiv.org/abs/1803.02155
3. Su, J. et al. (2021). *RoFormer: Enhanced Transformer with Rotary Position Embedding*. https://arxiv.org/abs/2104.09864
4. Kazemnejad, A. (blog). *Transformer Architecture: The Positional Encoding*. https://kazemnejad.com/blog/transformer_architecture_positional_encoding/
5. Saeed, M. (Machine Learning Mastery). *A Gentle Introduction to Positional Encoding in Transformer Models, Part 1*. https://machinelearningmastery.com/a-gentle-introduction-to-positional-encoding-in-transformer-models-part-1/
6. Suresh, S. K. (Towards Data Science). *Positional Embeddings in Transformers: A Math Guide to RoPE and ALiBi*. https://towardsdatascience.com/positional-embeddings-in-transformers-a-math-guide-to-rope-alibi/
7. Haviv, A. et al. (2022). *Transformer Language Models without Positional Encodings Still Learn Positional Information*. https://arxiv.org/abs/2203.16634